# Türkiye'de hava kirliliği: veri analizi
Veri: WHO Ambient Air Quality Database v8.0 (Haziran 2026), erişim tarihi: 12 Ekim 2026

## Kütüphaneler ve Yollar

In [1]:
import os

print("Çalışma klasörü:", os.getcwd())

YOL = "../data/raw/who-ambient-air-quality-database-version-2026-v8.xlsx"
KLASOR = "../data/processed"
os.makedirs(KLASOR, exist_ok=True)

print("Excel dosyası var mı:", os.path.exists(YOL))

Çalışma klasörü: C:\veri-hikayesi\notebooks
Excel dosyası var mı: True


In [2]:
import csv
from collections import Counter
import pandas as pd

ham = pd.read_excel(YOL, sheet_name=0, header=None, dtype=str).fillna("")
print("Excel boyutu:", ham.shape)

# Her satırdaki tüm hücreleri ; ile geri birleştir (bölünmeyi tersine çevirir)
satirlar = [";".join(r).rstrip(";") for r in ham.values.tolist()]

def ayir(satir):
    return next(csv.reader([satir.strip()]))

okunan = [ayir(s) for s in satirlar if s.strip()]
baslik = [b.strip() for b in okunan[0]]
n = len(baslik)

print("Satır başına sütun sayısı dağılımı:", Counter(len(r) for r in okunan[1:]).most_common(8))

govde, atilan = [], []
for r in okunan[1:]:
    if len(r) == n:
        govde.append(r)
    else:
        atilan.append(r)

print("Kullanılan satır:", len(govde))
print("Atılan satır:", len(atilan))

df = pd.DataFrame(govde, columns=baslik)
for c in df.columns:
    df[c] = df[c].str.strip()

sayisal = ["year", "pm10_concentration", "pm25_concentration", "no2_concentration",
           "pm10_tempcov", "pm25_tempcov", "no2_tempcov", "number_stations",
           "population", "latitude", "longitude"]
for c in sayisal:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(df.shape)
df.head(3)

Excel boyutu: (55221, 13)
Satır başına sütun sayısı dağılımı: [(21, 55211), (4, 3), (18, 3)]
Kullanılan satır: 55211
Atılan satır: 6
(55211, 21)


,who_region,iso3,country_name,city,year,version,pm10_concentration,pm25_concentration,no2_concentration,pm10_tempcov,...,no2_tempcov,type_of_stations,number_stations,reference,web_link,population,population_source,latitude,longitude,who_ms
0,EMR,AFG,Afghanistan,Kabul/AFG,2019.0,V5.0 (2022) (n=1),NaN,119.77,NaN,NaN,...,NaN,NA (n=1),1,"U.S. Department of State, United States Enviro...","[[[""EPA AirNow DOS"",""http://airnow.gov/index.c...",4221532.0,Imputed_from_UNWUP2018 (n=1),34.5358,69.1905,1
1,EUR,ALB,Albania,Durres/ALB,2014.0,V8.0 (2026) (n=1),13.85,7.74,36.74,40.0,...,9.0,Traffic-Urban (n=1),1,European Environmental Agency (n=1),https://discomap.eea.europa.eu/App/AQViewer/in...,NaN,NA (n=1),41.3199,19.4492,1
2,EUR,ALB,Albania,Durres/ALB,2015.0,V8.0 (2026) (n=1),17.65,12.60,26.63,79.0,...,84.0,Traffic-Urban (n=1),1,European Environmental Agency (n=1),https://discomap.eea.europa.eu/App/AQViewer/in...,NaN,NA (n=1),41.3199,19.4492,1


In [3]:
for r in atilan:
    print(len(r), r[:6])

4 ['WPR', 'KHM', 'Cambodia', 'Ta Khmau Cambodge']
18 ['Krong Ta Khmau Kandal Province/KHM"', '2019', 'V6.0  (2023) (n=1)', 'NA', '21', 'NA']
4 ['WPR', 'KHM', 'Cambodia', 'Ta Khmau Cambodge']
18 ['Krong Ta Khmau Kandal Province/KHM"', '2020', 'V6.0  (2023) (n=1)', 'NA', '21', 'NA']
4 ['WPR', 'KHM', 'Cambodia', 'Ta Khmau Cambodge']
18 ['Krong Ta Khmau Kandal Province/KHM"', '2021', 'V6.0  (2023) (n=1)', 'NA', '20', 'NA']


## Türkiye'yi Süzme

In [4]:
tr = df[df["iso3"] == "TUR"].copy()

print("Türkiye satır sayısı:", len(tr))
print("Şehir sayısı:", tr["city"].nunique())
print("Yıllar:", sorted(tr["year"].dropna().astype(int).unique()))
print("Sürümler:", tr["version"].value_counts().to_dict())
print("PM2.5 dolu satır:", tr["pm25_concentration"].notna().sum())
print("PM10 dolu satır:", tr["pm10_concentration"].notna().sum())

n25 = (tr.dropna(subset=["pm25_concentration"])
         .groupby("city")["year"].nunique().sort_values(ascending=False))
print(n25)
print((n25 >= 5).sum(), "şehirde en az 5 farklı yılda PM2.5 var")

tr[["city","year","version","pm10_concentration","pm25_concentration",
    "pm25_tempcov","type_of_stations"]].head(15)

Türkiye satır sayısı: 2200
Şehir sayısı: 256
Yıllar: [np.int64(2010), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Sürümler: {'V8.0 (2026) (n=1)': 1734, 'V8.0 (2026) (n=2)': 284, 'V8.0 (2026) (n=3)': 95, 'V4.0 (2018) (n=1)': 65, 'V8.0 (2026) (n=4)': 11, 'V4.0 (2018) (n=2)': 5, 'V4.0 (2018) (n=8)': 2, 'V4.0 (2018) (n=3)': 2, 'V4.0 (2018) (n=4)': 1, 'V4.0 (2018) (n=10)': 1}
PM2.5 dolu satır: 865
PM10 dolu satır: 2096
city
Istanbul/TUR    11
Silivri/TUR     11
Etlik/TUR       11
Gelibolu/TUR    11
Giresun/TUR     11
                ..
Tarsus/TUR       1
Aydin/TUR        1
Esenyurt/TUR     1
Atalar/TUR       1
Adalar/TUR       1
Name: year, Length: 128, dtype: int64
94 şehirde en az 5 farklı yılda PM2.5 var


,city,year,version,pm10_concentration,pm25_concentration,pm25_tempcov,type_of_stations
48650,Adalar/TUR,2016.0,V8.0 (2026) (n=1),15.520,3.54,96.0,Background-Rural (n=1)
48651,Adalar/TUR,2017.0,V8.0 (2026) (n=1),24.100,NaN,NaN,Background-Rural (n=1)
48652,Adalar/TUR,2018.0,V8.0 (2026) (n=1),23.370,NaN,NaN,Background-Rural (n=1)
48653,Adalar/TUR,2019.0,V8.0 (2026) (n=1),21.820,NaN,NaN,Background-Rural (n=1)
48654,Adalar/TUR,2020.0,V8.0 (2026) (n=1),17.640,NaN,NaN,Background-Rural (n=1)
48655,Adalar/TUR,2021.0,V8.0 (2026) (n=1),15.590,NaN,NaN,Background-Rural (n=1)
48656,Adalar/TUR,2022.0,V8.0 (2026) (n=1),15.790,NaN,NaN,Background-Rural (n=1)
48657,Adalar/TUR,2023.0,V8.0 (2026) (n=1),17.750,NaN,NaN,Background-Rural (n=1)
48658,Adalar/TUR,2024.0,V8.0 (2026) (n=1),21.400,NaN,NaN,Background-Rural (n=1)
48659,Adalar/TUR,2025.0,V8.0 (2026) (n=1),13.800,NaN,NaN,Background-Rural (n=1)


In [5]:
p = tr.dropna(subset=["pm25_concentration"]).copy()
p["year"] = p["year"].astype(int)
p["surum"] = p["version"].str.extract(r"(V\d\.\d)")

print("PM2.5 satırları yıla göre:")
print(p.groupby("year").size().to_string())
print("\nSürüme göre:", p["surum"].value_counts().to_dict())

cift = p.groupby(["city", "year"]).size()
print("\nAynı yerleşim+yıl için birden fazla satır:", (cift > 1).sum())

print("\npm25_tempcov (zaman kapsaması %) özeti:")
print(p["pm25_tempcov"].describe().round(1).to_string())
print("kapsama < 75:", (p["pm25_tempcov"] < 75).sum(),
      "| boş:", p["pm25_tempcov"].isna().sum())

ozet = p.groupby("city")["year"].agg(["nunique", "min", "max"])
ozet["aralik"] = ozet["max"] - ozet["min"]
uygun = ozet[(ozet["nunique"] >= 5) & (ozet["aralik"] >= 5)]
print("\n≥5 yıl VE ilk-son yıl arası ≥5 yıl:", len(uygun))
print("2014 veya öncesinde PM2.5 verisi olan yerleşim:", (ozet["min"] <= 2014).sum())
ozet.sort_values("nunique", ascending=False).head(15)

PM2.5 satırları yıla göre:
year
2015     33
2016     36
2017     33
2018     65
2019     83
2020    102
2021    102
2022    103
2023    116
2024    100
2025     92

Sürüme göre: {'V8.0': 865}

Aynı yerleşim+yıl için birden fazla satır: 0

pm25_tempcov (zaman kapsaması %) özeti:
count    773.0
mean      85.0
std       21.8
min        1.0
25%       85.0
50%       94.0
75%       97.0
max      100.0
kapsama < 75: 142 | boş: 92

≥5 yıl VE ilk-son yıl arası ≥5 yıl: 91
2014 veya öncesinde PM2.5 verisi olan yerleşim: 0


,nunique,min,max,aralik
city,,,,
Istanbul/TUR,11,2015,2025,10
Silivri/TUR,11,2015,2025,10
Etlik/TUR,11,2015,2025,10
Gelibolu/TUR,11,2015,2025,10
Giresun/TUR,11,2015,2025,10
Gorukle/TUR,11,2015,2025,10
Ilkadim/TUR,11,2015,2025,10
Edirne/TUR,11,2015,2025,10
Incirli/TUR,11,2015,2025,10


In [6]:
t = tr.dropna(subset=["pm25_concentration"]).copy()
t["year"] = t["year"].astype(int)
t["yerlesim"] = t["city"].str.replace("/TUR", "", regex=False)

# 1) Kapsama durumu
t["kapsama_durum"] = "yeterli"
t.loc[t["pm25_tempcov"].isna(), "kapsama_durum"] = "bilinmiyor"
t.loc[t["pm25_tempcov"] < 75, "kapsama_durum"] = "dusuk"
print("Kapsama durumu:", t["kapsama_durum"].value_counts().to_dict())

# 2) 2025 yılı kapsaması
print("\n2025 kapsama özeti:")
print(t[t["year"] == 2025]["pm25_tempcov"].describe().round(1).to_string())
print("Yıla göre ortalama kapsama:")
print(t.groupby("year")["pm25_tempcov"].mean().round(1).to_string())

# 3) İstasyon türleri
print("\nİstasyon türü:")
print(t["type_of_stations"].str.replace(r"\s*\(n=\d+\)", "", regex=True).value_counts().to_string())

# 4) Temiz küme
temiz = t[t["kapsama_durum"] == "yeterli"].copy()
print("\nTemiz satır:", len(temiz), "| yerleşim:", temiz["yerlesim"].nunique())

# 5) Uç değerler
print("\nEn yüksek 8 değer:")
print(temiz.nlargest(8, "pm25_concentration")[["yerlesim", "year", "pm25_concentration", "pm25_tempcov"]].to_string(index=False))
print("En düşük 5 değer:")
print(temiz.nsmallest(5, "pm25_concentration")[["yerlesim", "year", "pm25_concentration", "pm25_tempcov"]].to_string(index=False))

# 6) Uygun yerleşimler: ≥5 yıl ve ilk-son arası ≥5 yıl
oz = temiz.groupby("yerlesim")["year"].agg(["nunique", "min", "max"])
oz["aralik"] = oz["max"] - oz["min"]
uygun = oz[(oz["nunique"] >= 5) & (oz["aralik"] >= 5)].index
print("\nUygun yerleşim sayısı:", len(uygun))
sec = temiz[temiz["yerlesim"].isin(uygun)]
print("Yıla göre uygun yerleşim sayısı:")
print(sec.groupby("year")["yerlesim"].nunique().to_string())
print("Geçerli yıl sayısı dağılımı:", oz.loc[uygun, "nunique"].value_counts().sort_index().to_dict())

Kapsama durumu: {'yeterli': 631, 'dusuk': 142, 'bilinmiyor': 92}

2025 kapsama özeti:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Yıla göre ortalama kapsama:
year
2015    72.7
2016    98.2
2017    94.7
2018    65.9
2019    82.4
2020    82.0
2021    86.5
2022    92.0
2023    86.3
2024    88.5
2025     NaN

İstasyon türü:
type_of_stations
Background-Urban                                     429
Background-Urban; Traffic-Urban                      156
Background-Rural                                      85
Traffic-Urban                                         77
Background-Urban; Industrial-Urban                    24
Industrial-Urban                                      22
Background-Suburban                                   18
Industrial-Suburban                                   17
Background-Urban; Industrial-Urban; Traffic-Urban     17
Background-Urban; Industrial-Suburban                  8
Background,Traffic-Urban; Backg

## Veri Setini Temizleme

In [7]:
import numpy as np

a = temiz[(temiz["year"] >= 2015) & (temiz["year"] <= 2024)].copy()
print("2015–2024 temiz satır:", len(a), "| yerleşim:", a["yerlesim"].nunique())

# A: uzun seri (2015-2024 içinde en az 8 yıl)
yil_sayisi = a.groupby("yerlesim")["year"].nunique()
A_ad = yil_sayisi[yil_sayisi >= 8].index
A = a[a["yerlesim"].isin(A_ad)]
print("\nA kümesi (>=8 yıl):", len(A_ad), "yerleşim")
print(A.groupby("year")["yerlesim"].nunique().to_dict())

# B: geniş karşılaştırma (2019-2024 içinde en az 5 yıl)
b = a[a["year"] >= 2019]
y6 = b.groupby("yerlesim")["year"].nunique()
B_ad = y6[y6 >= 5].index
B = b[b["yerlesim"].isin(B_ad)]
print("\nB kümesi (2019-2024, >=5 yıl):", len(B_ad), "yerleşim")
print(B.groupby("year")["yerlesim"].nunique().to_dict())
print("A ve B ortak yerleşim:", len(set(A_ad) & set(B_ad)))

# B için eğim (µg/m³ / yıl)
rows = []
for ad, g in B.groupby("yerlesim"):
    g = g.sort_values("year")
    rows.append({"yerlesim": ad,
                 "egim": np.polyfit(g["year"], g["pm25_concentration"], 1)[0],
                 "n_yil": g["year"].nunique(),
                 "ilk_yil": int(g["year"].iloc[0]), "ilk": g["pm25_concentration"].iloc[0],
                 "son_yil": int(g["year"].iloc[-1]), "son": g["pm25_concentration"].iloc[-1]})
trend = pd.DataFrame(rows)

print("\nEğim özeti (µg/m³/yıl):")
print(trend["egim"].describe().round(2).to_string())

print("\nEşik duyarlılığı (düşen / değişmeyen / yükselen):")
for esik in [0.3, 0.5, 1.0]:
    print(esik, (trend.egim < -esik).sum(),
          (trend.egim.abs() <= esik).sum(), (trend.egim > esik).sum())

print("\nEn çok düşen 5:")
print(trend.sort_values("egim").head(5).round(2).to_string(index=False))
print("En çok yükselen 5:")
print(trend.sort_values("egim").tail(5).round(2).to_string(index=False))

son24 = B[B["year"] == 2024]["pm25_concentration"]
print("\n2024'te B kümesinde yerleşim sayısı:", len(son24),
      "| WHO rehberi 5 µg/m³'ün üstünde olan:", (son24 > 5).sum())

2015–2024 temiz satır: 631 | yerleşim: 118

A kümesi (>=8 yıl): 26 yerleşim
{2015: 18, 2016: 26, 2017: 25, 2018: 21, 2019: 22, 2020: 26, 2021: 26, 2022: 26, 2023: 22, 2024: 23}

B kümesi (2019-2024, >=5 yıl): 65 yerleşim
{2019: 52, 2020: 63, 2021: 64, 2022: 65, 2023: 63, 2024: 64}
A ve B ortak yerleşim: 23

Eğim özeti (µg/m³/yıl):
count    65.00
mean     -0.92
std       1.39
min      -6.88
25%      -1.43
50%      -0.70
75%      -0.12
max       1.02

Eşik duyarlılığı (düşen / değişmeyen / yükselen):
0.3 44 11 10
0.5 39 18 8
1.0 23 40 2

En çok düşen 5:
yerlesim  egim  n_yil  ilk_yil   ilk  son_yil   son
 Erzurum -6.88      6     2019 53.78     2024 18.85
 Caycuma -4.94      5     2020 23.64     2024  6.91
   Corum -3.97      5     2020 37.45     2024 21.26
Tavsanli -3.20      5     2020 23.12     2024 11.89
 Karabuk -3.05      6     2019 22.14     2024  9.69
En çok yükselen 5:
  yerlesim  egim  n_yil  ilk_yil   ilk  son_yil   son
   Icmeler  0.75      6     2019 18.13     2024 17.82
  T

## Keşifsel Veri Analizi

In [8]:
import os

KLASOR = r"C:\veri-hikayesi\data\processed"
os.makedirs(KLASOR, exist_ok=True)
ESIK = 0.5   # µg/m³/yıl, Yöntem'de gerekçelendirilecek

# 1) yerleşim-yıl tablosu
cy = a[["yerlesim", "year", "pm25_concentration", "pm25_tempcov",
        "type_of_stations", "latitude", "longitude"]].copy()
cy.columns = ["yerlesim", "yil", "pm25", "kapsama", "istasyon_turu", "enlem", "boylam"]
cy["istasyon_turu"] = cy["istasyon_turu"].str.replace(r"\s*\(n=\d+\)", "", regex=True)
cy["kume_A"] = cy["yerlesim"].isin(A_ad)
cy["kume_B"] = cy["yerlesim"].isin(B_ad)
cy = cy.sort_values(["yerlesim", "yil"]).round(2)
cy.to_csv(os.path.join(KLASOR, "city_year_pm25.csv"), index=False, encoding="utf-8-sig")

# 2) eğim tablosu (B kümesi)
def grup(e):
    if e < -ESIK: return "dusen"
    if e > ESIK:  return "yukselen"
    return "degismeyen"

tr_t = trend.copy()
tr_t["grup"] = tr_t["egim"].apply(grup)
konum = B.groupby("yerlesim")[["latitude", "longitude"]].first().reset_index()
konum.columns = ["yerlesim", "enlem", "boylam"]
v24 = (B[B["year"] == 2024][["yerlesim", "pm25_concentration"]]
       .rename(columns={"pm25_concentration": "pm25_2024"}))
tr_t = tr_t.merge(konum, on="yerlesim", how="left").merge(v24, on="yerlesim", how="left")
tr_t = tr_t.sort_values("egim").round(2)
tr_t.to_csv(os.path.join(KLASOR, "city_trend.csv"), index=False, encoding="utf-8-sig")

# 3) ulusal özet (tüm temiz / A / B)
ulusal = pd.DataFrame({"yil": range(2015, 2025)})
for ad, d in [("tum", a), ("A", A), ("B", B)]:
    g = d.groupby("year")["pm25_concentration"].agg(["median", "mean", "size"]).round(2)
    g.columns = [f"{ad}_medyan", f"{ad}_ortalama", f"{ad}_n"]
    ulusal = ulusal.merge(g, left_on="yil", right_index=True, how="left")
ulusal.to_csv(os.path.join(KLASOR, "national_summary.csv"), index=False, encoding="utf-8-sig")

# 4) kapsama tablosu (veri boşlukları adımı için)
kap = (t[(t["year"] >= 2015) & (t["year"] <= 2024)]
       .groupby(["year", "kapsama_durum"]).size().unstack(fill_value=0).reset_index())
kap = kap.rename(columns={"year": "yil"})
kap.to_csv(os.path.join(KLASOR, "coverage_by_year.csv"), index=False, encoding="utf-8-sig")

# Kontroller
print("Grup sayıları (eşik ±%s):" % ESIK, tr_t["grup"].value_counts().to_dict())
print("\nWHO yorumu için 2024 değer aralıkları (B kümesi):")
b24 = B[B["year"] == 2024]["pm25_concentration"]
print(pd.cut(b24, [0, 5, 10, 15, 25, 35, 100]).value_counts().sort_index().to_string())
print("\nDosyalar:")
for f in os.listdir(KLASOR):
    print(f, round(os.path.getsize(os.path.join(KLASOR, f)) / 1024, 1), "KB")
print()
print(ulusal.to_string(index=False))

Grup sayıları (eşik ±0.5): {'dusen': 39, 'degismeyen': 18, 'yukselen': 8}

WHO yorumu için 2024 değer aralıkları (B kümesi):
pm25_concentration
(0, 5]        0
(5, 10]       8
(10, 15]     14
(15, 25]     39
(25, 35]      3
(35, 100]     0

Dosyalar:
city_trend.csv 4.1 KB
city_year_pm25.csv 42.5 KB
coverage_by_year.csv 0.1 KB
national_summary.csv 0.5 KB
snapshot_2024.csv 5.6 KB

 yil  tum_medyan  tum_ortalama  tum_n  A_medyan  A_ortalama  A_n  B_medyan  B_ortalama  B_n
2015       22.85         24.52     22     22.85       25.03   18       NaN         NaN  NaN
2016       18.34         20.39     36     19.24       22.09   26       NaN         NaN  NaN
2017       23.08         25.40     31     23.11       26.03   25       NaN         NaN  NaN
2018       22.39         22.82     33     22.72       22.94   21       NaN         NaN  NaN
2019       21.25         21.78     62     21.25       21.30   22     20.78       21.41 52.0
2020       19.27         20.07     79     18.22       20.68   26  

## Kümeler ve Eğim

In [9]:
trend["fark"] = trend["son"] - trend["ilk"]
print("İlk yıl dağılımı:", trend["ilk_yil"].value_counts().to_dict())
print("Son yıl dağılımı:", trend["son_yil"].value_counts().to_dict())
print("Çiftli fark, medyan:", round(trend["fark"].median(), 2),
      "| ortalama:", round(trend["fark"].mean(), 2))
print("Son değeri ilkinden düşük olan:", (trend["fark"] < 0).sum(), "/", len(trend))

b24 = B[B["year"] == 2024].sort_values("pm25_concentration")
print("\n2024'te rehbere en yakın 8:")
print(b24.head(8)[["yerlesim", "pm25_concentration", "type_of_stations"]].to_string(index=False))
print("2024'te en yüksek 5:")
print(b24.tail(5)[["yerlesim", "pm25_concentration", "type_of_stations"]].to_string(index=False))

print("\nErzurum ve Iğdır:")
print(a[a["yerlesim"].isin(["Erzurum", "Igdir"])]
      [["yerlesim", "year", "pm25_concentration", "pm25_tempcov"]]
      .sort_values(["yerlesim", "year"]).to_string(index=False))

print("\nKapsama tablosu:")
print(pd.read_csv(KLASOR + r"\coverage_by_year.csv").to_string(index=False))

İlk yıl dağılımı: {2019: 52, 2020: 13}
Son yıl dağılımı: {2024: 64, 2023: 1}
Çiftli fark, medyan: -3.44 | ortalama: -4.56
Son değeri ilkinden düşük olan: 54 / 65

2024'te rehbere en yakın 8:
    yerlesim  pm25_concentration                               type_of_stations
      Kumkoy                5.91                         Background-Rural (n=1)
     Caycuma                6.91                         Background-Urban (n=1)
    Kirsehir                7.56                         Background-Urban (n=1)
     Cankiri                8.44                         Background-Urban (n=1)
    Batikent                8.52 Background-Urban (n=1); Industrial-Urban (n=1)
     Incirli                9.28                         Background-Urban (n=1)
     Karabuk                9.69                         Industrial-Urban (n=1)
Bahcelievler                9.81    Background-Urban (n=2); Traffic-Urban (n=1)
2024'te en yüksek 5:
yerlesim  pm25_concentration                               type_of_s

## Bulgular

In [10]:
t24 = t[t["year"] == 2024].copy()
print("2024'te temiz ölçümü olan yerleşim (tümü):", len(t24))
print("Bunlardan B kümesinde olmayan:", (~t24["yerlesim"].isin(B_ad)).sum())
print("WHO 5 üstü olan:", (t24["pm25_concentration"] > 5).sum(), "/", len(t24))
print("En düşük 3:", t24.nsmallest(3, "pm25_concentration")[["yerlesim", "pm25_concentration"]].values.tolist())

dis = a[~a["yerlesim"].isin(B_ad)]
print("\nB dışında kalıp ≥2 yıllık verisi olan en yüksek yerleşimler:")
print(dis.groupby("yerlesim")["pm25_concentration"].agg(["mean", "max", "count"])
        .query("count >= 2").sort_values("mean", ascending=False).head(8).round(1).to_string())

2024'te temiz ölçümü olan yerleşim (tümü): 100
Bunlardan B kümesinde olmayan: 35
WHO 5 üstü olan: 99 / 100
En düşük 3: [['Kuyulusebil', 3.62], ['Kumkoy', 5.91], ['Agli', 6.18]]

B dışında kalıp ≥2 yıllık verisi olan en yüksek yerleşimler:
          mean   max  count
yerlesim                   
Igdir     54.7  67.8      4
Osmaniye  38.8  39.5      2
Kesan     38.6  66.1      8
Akyazi    36.7  39.8      5
Duzce     34.0  44.6      3
Cankaya   27.5  29.5      3
Elbistan  27.3  29.6      3
Hidirbey  27.1  27.8      2


In [11]:
# Seçili yerleşimlerin yıllara göre serisi
for ad in ["Kesan", "Igdir", "Duzce", "Akyazi", "Osmaniye", "Kuyulusebil"]:
    g = a[a["yerlesim"] == ad].sort_values("year")
    print(ad, "->", [(int(y), round(v, 1)) for y, v in zip(g["year"], g["pm25_concentration"])])
    print("   istasyon:", g["type_of_stations"].iloc[0] if len(g) else "-")

# B içi ve B dışı 2024 karşılaştırması
t24["kume_B"] = t24["yerlesim"].isin(B_ad)
print("\n2024 medyan - B içi:", round(t24[t24.kume_B]["pm25_concentration"].median(), 1),
      "| B dışı:", round(t24[~t24.kume_B]["pm25_concentration"].median(), 1))
print("2024 medyan - tümü:", round(t24["pm25_concentration"].median(), 1))
print("2024'te 25 µg/m³ üstü: B içi", (t24[t24.kume_B]["pm25_concentration"] > 25).sum(),
      "| B dışı", (t24[~t24.kume_B]["pm25_concentration"] > 25).sum())

# Dot plot için 2024 görünümü (100 yerleşim)
snap = t24[["yerlesim", "pm25_concentration", "pm25_tempcov", "type_of_stations",
            "latitude", "longitude", "kume_B"]].copy()
snap.columns = ["yerlesim", "pm25", "kapsama", "istasyon_turu", "enlem", "boylam", "kume_B"]
snap["istasyon_turu"] = snap["istasyon_turu"].str.replace(r"\s*\(n=\d+\)", "", regex=True)
snap["rehber_ustu"] = snap["pm25"] > 5
snap = snap.sort_values("pm25").round(2)
snap.to_csv(KLASOR + r"\snapshot_2024.csv", index=False, encoding="utf-8-sig")
print("\nsnapshot_2024.csv yazıldı:", len(snap), "satır")

Kesan -> [(2015, 63.8), (2016, 45.4), (2017, 66.1), (2018, 39.9), (2019, 31.9), (2020, 29.8), (2021, 20.2), (2022, 11.9)]
   istasyon: Background-Urban (n=1)
Igdir -> [(2021, 67.8), (2022, 52.9), (2023, 48.0), (2024, 50.3)]
   istasyon: Background-Urban (n=1)
Duzce -> [(2019, 44.6), (2023, 30.0), (2024, 27.3)]
   istasyon: Background-Urban (n=1)
Akyazi -> [(2018, 39.8), (2019, 35.1), (2020, 35.8), (2021, 38.4), (2022, 34.4)]
   istasyon: Industrial-Suburban (n=1)
Osmaniye -> [(2022, 39.5), (2023, 38.2)]
   istasyon: Background-Urban (n=1)
Kuyulusebil -> [(2023, 3.2), (2024, 3.6)]
   istasyon: Background-Rural (n=1)

2024 medyan - B içi: 17.4 | B dışı: 15.7
2024 medyan - tümü: 16.7
2024'te 25 µg/m³ üstü: B içi 3 | B dışı 5

snapshot_2024.csv yazıldı: 100 satır


In [12]:
t24 = temiz[temiz["year"] == 2024].copy()          # YALNIZCA yeterli kapsamalı
t24["kume_B"] = t24["yerlesim"].isin(B_ad)
ic, dis = t24[t24.kume_B], t24[~t24.kume_B]

print("2024, kapsaması yeterli yerleşim:", len(t24), "| B içi:", len(ic), "| B dışı:", len(dis))
print("WHO 5 üstü:", (t24.pm25_concentration > 5).sum(), "/", len(t24))
print("5 ve altı olanlar:")
print(t24[t24.pm25_concentration <= 5][["yerlesim", "pm25_concentration", "pm25_tempcov", "type_of_stations"]].to_string(index=False))

print("\nMedyan: tümü", round(t24.pm25_concentration.median(), 1),
      "| B içi", round(ic.pm25_concentration.median(), 1),
      "| B dışı", round(dis.pm25_concentration.median(), 1))
print("25 üstü: B içi", (ic.pm25_concentration > 25).sum(), "/", len(ic),
      "| B dışı", (dis.pm25_concentration > 25).sum(), "/", len(dis))
print("\nB dışı, 2024'te en yüksek 5:")
print(dis.nlargest(5, "pm25_concentration")[["yerlesim", "pm25_concentration"]].to_string(index=False))

snap = t24[["yerlesim", "pm25_concentration", "pm25_tempcov", "type_of_stations",
            "latitude", "longitude", "kume_B"]].copy()
snap.columns = ["yerlesim", "pm25", "kapsama", "istasyon_turu", "enlem", "boylam", "kume_B"]
snap["istasyon_turu"] = snap["istasyon_turu"].str.replace(r"\s*\(n=\d+\)", "", regex=True)
snap["rehber_ustu"] = snap["pm25"] > 5
snap.sort_values("pm25").round(2).to_csv(KLASOR + r"\snapshot_2024.csv", index=False, encoding="utf-8-sig")
print("\nsnapshot_2024.csv yeniden yazıldı:", len(snap), "satır")

2024, kapsaması yeterli yerleşim: 88 | B içi: 64 | B dışı: 24
WHO 5 üstü: 87 / 88
5 ve altı olanlar:
   yerlesim  pm25_concentration  pm25_tempcov       type_of_stations
Kuyulusebil                3.62          95.0 Background-Rural (n=1)

Medyan: tümü 16.6 | B içi 17.3 | B dışı 14.5
25 üstü: B içi 3 / 64 | B dışı 2 / 24

B dışı, 2024'te en yüksek 5:
yerlesim  pm25_concentration
   Igdir               50.28
   Duzce               27.28
Selcuklu               21.58
  Levent               21.35
    Kars               19.68

snapshot_2024.csv yeniden yazıldı: 88 satır


## Dışa Aktarım

In [13]:
import shutil, glob, os

SITE_DATA = r"C:\veri-hikayesi\site\data"
os.makedirs(SITE_DATA, exist_ok=True)
os.makedirs(r"C:\veri-hikayesi\site\css", exist_ok=True)
os.makedirs(r"C:\veri-hikayesi\site\js", exist_ok=True)

for f in glob.glob(KLASOR + r"\*.csv"):
    shutil.copy(f, SITE_DATA)

# Ham veriyi de data/raw içine koy
RAW = r"C:\veri-hikayesi\data\raw"
os.makedirs(RAW, exist_ok=True)
for f in glob.glob(os.path.expanduser(r"~\Downloads\who-ambient-air-quality-database-version-2026-v8.xlsx")):
    shutil.copy(f, RAW)

print(os.listdir(SITE_DATA))
print(os.listdir(RAW))

['city_trend.csv', 'city_year_pm25.csv', 'coverage_by_year.csv', 'national_summary.csv', 'snapshot_2024.csv']
['who-ambient-air-quality-database-version-2026-v8 (1).xlsx', 'who-ambient-air-quality-database-version-2026-v8.xlsx']


In [14]:
path = "C:/Users/LENOVO/OneDrive/Desktop/who_air_quality.csv"

with open(path, "rb") as f:
    ilk = f.read(300)
print(ilk)

b'"who_region,""iso3"",""country_name"",""city"",""year"",""version"",""pm10_concentration"",""pm25_concentration"",""no2_concentration"",""pm10_tempcov"",""pm25_tempcov"",""no2_tempcov"",""type_of_stations"",""number_stations"",""reference"",""web_link"",""population"",""population_source"",""latitud'
